In [1]:
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output, State
import plotly.graph_objects as go
import plotly.express as px
from dash import no_update

# 创建应用
app = dash.Dash(__name__)
app.config.suppress_callback_exceptions = True

# 读取汽车销量数据（本地文件）
data = pd.read_csv('Datasets/automobile-sales.csv')

# 年份下拉框的候选列表（1980~2023）
year_list = [i for i in range(1980, 2024, 1)]

print('数据准备完成：', data.shape)
print('年份范围：', year_list[0], '-', year_list[-1])

数据准备完成： (2112, 15)
年份范围： 1980 - 2023


In [2]:
app.layout = html.Div(children=[
    # TASK 2.1：标题
    html.H1('Automobile Sales Statistics Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 24}),

    # TASK 2.2（第一个下拉框）：报告类型
    html.Div([
        html.H2('Select Statistics:', style={'margin-right': '2em'}),
        dcc.Dropdown(id='dropdown-statistics',
                     options=[
                         {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
                         {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
                     ],
                     placeholder='Select a report type',
                     value='Select Statistics',
                     style={'width': '80%', 'padding': '3px',
                            'font-size': '20px', 'textAlignLast': 'center'})
    ]),

    # TASK 2.2（第二个下拉框）：年份
    html.Div([
        html.H2('Select Year:', style={'margin-right': '2em'}),
        dcc.Dropdown(id='select-year',
                     options=[{'label': i, 'value': i} for i in year_list],
                     placeholder='Select-year',
                     # 说明：题目写的是 value='Select-year'。这里改成 None（初始不预选），
                     # 否则年份列里会拿字符串去筛选、导致图表为空。placeholder 已经负责显示"Select-year"。
                     value=None,
                     style={'width': '80%', 'padding': '3px',
                            'font-size': '20px', 'textAlignLast': 'center'})
    ]),

    # TASK 2.3：输出容器（4 张图会塞进这里，2 行 2 列）
    html.Div([
        html.Div(id='output-container', className='chart-grid', style={'display': 'flex'}),
    ])
])

print('布局创建完成！')

布局创建完成！


In [3]:
# TASK 2.4（回调 1）
@app.callback(
    Output(component_id='select-year', component_property='disabled'),
    Input(component_id='dropdown-statistics', component_property='value'))
def update_input_container(selected_statistics):
    if selected_statistics == 'Yearly Statistics':
        return False     # 年度统计 → 启用年份下拉框
    else:
        return True      # 衰退期统计 → 禁用年份下拉框

In [4]:
# TASK 2.4（回调 2）+ TASK 2.5 / 2.6
@app.callback(
    Output(component_id='output-container', component_property='children'),
    [Input(component_id='dropdown-statistics', component_property='value'),
     Input(component_id='select-year', component_property='value')])
def update_output_container(selected_statistics, input_year):

    # ==================== 情况一：衰退期统计 ====================
    if selected_statistics == 'Recession Period Statistics':
        recession_data = data[data['Recession'] == 1]

        # 图 1：衰退期逐年平均销量（折线）
        yearly_rec = recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        R_chart1 = dcc.Graph(figure=px.line(
            yearly_rec, x='Year', y='Automobile_Sales',
            title='Average Automobile Sales fluctuation over Recession Period'))

        # 图 2：各车型平均销量（柱状）
        average_sales = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        R_chart2 = dcc.Graph(figure=px.bar(
            average_sales, x='Vehicle_Type', y='Automobile_Sales',
            title='Average Number of Vehicles Sold by Vehicle Type'))

        # 图 3：各车型广告支出占比（饼图）
        exp_rec = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        R_chart3 = dcc.Graph(figure=px.pie(
            exp_rec, values='Advertising_Expenditure', names='Vehicle_Type',
            title='Total Expenditure Share by Vehicle Type during Recessions'))

        # 图 4：失业率对各车型销量的影响（柱状，按车型分色）
        unemp_data = recession_data.groupby(['unemployment_rate', 'Vehicle_Type'])['Automobile_Sales'].mean().reset_index()
        R_chart4 = dcc.Graph(figure=px.bar(
            unemp_data, x='unemployment_rate', y='Automobile_Sales', color='Vehicle_Type',
            labels={'unemployment_rate': 'Unemployment Rate',
                    'Automobile_Sales': 'Average Automobile Sales'},
            title='Effect of Unemployment Rate on Vehicle Type and Sales'))

        # 2 行 2 列返回
        return [
            html.Div(className='chart-item',
                     children=[html.Div(children=R_chart1), html.Div(children=R_chart2)],
                     style={'display': 'flex'}),
            html.Div(className='chart-item',
                     children=[html.Div(children=R_chart3), html.Div(children=R_chart4)],
                     style={'display': 'flex'})
        ]

    # ==================== 情况二：年度统计 ====================
    elif (input_year and selected_statistics == 'Yearly Statistics'):
        yearly_data = data[data['Year'] == int(input_year)]

        # 图 1：全期逐年平均销量（折线）
        yas = data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        Y_chart1 = dcc.Graph(figure=px.line(
            yas, x='Year', y='Automobile_Sales',
            title='Yearly Automobile Sales for the Whole Period'))

        # 图 2：该年逐月总销量（折线）
        mas = data.groupby('Month')['Automobile_Sales'].sum().reset_index()
        Y_chart2 = dcc.Graph(figure=px.line(
            mas, x='Month', y='Automobile_Sales',
            title='Total Monthly Automobile Sales'))

        # 图 3：该年各车型平均销量（柱状）
        avr_vdata = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        Y_chart3 = dcc.Graph(figure=px.bar(
            avr_vdata, x='Vehicle_Type', y='Automobile_Sales',
            title='Average Vehicles Sold by Vehicle Type in the year {}'.format(input_year)))

        # 图 4：该年各车型广告支出（饼图）
        exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        Y_chart4 = dcc.Graph(figure=px.pie(
            exp_data, values='Advertising_Expenditure', names='Vehicle_Type',
            title='Total Advertisement Expenditure for Each Vehicle'))

        return [
            html.Div(className='chart-item',
                     children=[html.Div(children=Y_chart1), html.Div(children=Y_chart2)],
                     style={'display': 'flex'}),
            html.Div(className='chart-item',
                     children=[html.Div(children=Y_chart3), html.Div(children=Y_chart4)],
                     style={'display': 'flex'})
        ]
    else:
        return html.Div('Please select a report type.')

In [5]:
try:
    app.run(jupyter_mode='inline')
    print('Dash 应用已启动（notebook 内联模式）。')
except Exception as e:
    print('内联启动受限（不影响代码本身正确性）：', e)

Dash 应用已启动（notebook 内联模式）。


In [6]:
def _collect_figures(node):
    """从 Dash 组件树里把所有图（dcc.Graph 的 figure）收集出来。
    仅用于在 notebook 里"预览"回调结果。"""
    figs = []
    # 情况一：本身就是一个列表/元组（回调常返回列表），逐个往里找
    if isinstance(node, (list, tuple)):
        for item in node:
            figs += _collect_figures(item)
        return figs
    # 情况二：本身是 dcc.Graph，直接取它的 figure
    if getattr(node, 'figure', None) is not None:
        figs.append(node.figure)
    # 情况三：是 html.Div 之类的容器，往下找它的 children
    ch = getattr(node, 'children', None)
    if ch is not None:
        if not isinstance(ch, (list, tuple)):
            ch = [ch]
        for c in ch:
            figs += _collect_figures(c)
    return figs

graphs_rec = update_output_container('Recession Period Statistics', None)
print('衰退期报告共', len(_collect_figures(graphs_rec)), '张图：')
for i, f in enumerate(_collect_figures(graphs_rec), 1):
    print('--- 图', i, '---')
    f.show()

衰退期报告共 4 张图：
--- 图 1 ---


--- 图 2 ---


--- 图 3 ---


--- 图 4 ---


In [7]:
graphs_year = update_output_container('Yearly Statistics', 2010)
print('年度报告（2010）共', len(_collect_figures(graphs_year)), '张图：')
for i, f in enumerate(_collect_figures(graphs_year), 1):
    print('--- 图', i, '---')
    f.show()

年度报告（2010）共 4 张图：
--- 图 1 ---


--- 图 2 ---


--- 图 3 ---


--- 图 4 ---
